# Weight Decay Example

In [4]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "online_fdr").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))
        break

In [ ]:
from online_fdr.p_values.investing.lord.mem_decay import LORDMemoryDecay
from online_fdr.core.utils.evaluation import MemoryDecayFDR, calculate_power, calculate_sfdr
from online_fdr.core.utils.format import format_result
from online_fdr.core.utils.generation import DataGenerator, GaussianLocationModel

N = 500
dgp = GaussianLocationModel(alt_mean=3.0, alt_std=1.0, one_sided=True)
generator = DataGenerator(
    n=N, pi0=0.9, dgp=dgp
)  # pi0 = 1 - contamination = 1 - 0.1 = 0.9
mem_decay_lord = LORDMemoryDecay(alpha=0.05, delta=0.99, eta=0.01)
false_positive = 0
true_positive = 0
false_negatives = 0
mem_fdr = MemoryDecayFDR(delta=0.99, offset=0)
current_mem_fdr = 0.0

In [5]:
for i in range(0, N):
    p_value, label = generator.sample_one()  # sample generation
    result = mem_decay_lord.test_one(p_value)  # LORD memory decay

    true_positive += label and result
    false_positive += not label and result
    false_negatives += label and not result

    current_mem_fdr = mem_fdr.score_one(result, label)
    if i < 25:
        format_result(i, result, p_value, mem_decay_lord.alpha)

[1] False (0.273 > 0.000027)
[2] False (0.019 > 0.000006)
[3] False (0.376 > 0.000005)
[4] False (0.004 > 0.000005)
[5] False (0.082 > 0.000005)
[6] False (0.693 > 0.000005)
[7] False (0.869 > 0.000005)
[8] False (0.980 > 0.000005)
[9] False (0.001 > 0.000005)
[10] False (0.205 > 0.000005)
[11] False (0.205 > 0.000005)
[12] False (0.600 > 0.000005)
[13] False (0.915 > 0.000005)
[14] False (0.326 > 0.000005)
[15] False (0.749 > 0.000005)
[16] False (0.338 > 0.000005)
[17] False (0.110 > 0.000005)
[18] False (0.014 > 0.000005)
[19] False (0.858 > 0.000005)
[20] False (0.184 > 0.000005)
[21] False (0.213 > 0.000005)
[22] False (0.006 > 0.000005)
[23] False (0.567 > 0.000005)
[24] False (0.739 > 0.000005)
[25] False (0.704 > 0.000005)


In [6]:
print(f"Empirical sFDR: {calculate_sfdr(tp=true_positive, fp=false_positive)}")
print(f"Empirical Power: {calculate_power(tp=true_positive, fn=false_negatives)}")
print(f"Final memory-decay FDR: {current_mem_fdr}")

Empirical sFDR: 0.0
Empirical Power: 0.26
Final memory-decay FDR: 0.0
